In [2]:
!pip install -q pypdf faiss-cpu sentence-transformers

In [3]:
import pypdf
import faiss
import sentence_transformers

print("All required packages are working!")

All required packages are working!


In [4]:
from google.colab import files

uploaded = files.upload()

print("Uploaded file:", list(uploaded.keys()))

Saving English_Booklet_2.pdf to English_Booklet_2.pdf
Uploaded file: ['English_Booklet_2.pdf']


In [6]:
from pypdf import PdfReader

pdf_name = list(uploaded.keys())[0]
reader = PdfReader(pdf_name)

print("PDF name:", pdf_name)
print("Number of pages:", len(reader.pages))

PDF name: English_Booklet_2.pdf
Number of pages: 168


In [7]:
from pypdf import PdfReader

pdf_name = list(uploaded.keys())[0]
reader = PdfReader(pdf_name)

documents = []

for page_number, page in enumerate(reader.pages):
    text = page.extract_text()

    if text and text.strip():
        documents.append({
            "page": page_number + 1,
            "text": text.strip()
        })

print("Total PDF pages:", len(reader.pages))
print("Pages containing text:", len(documents))

print("\n--- First page preview ---\n")
print(documents[0]["text"][:2000])

Total PDF pages: 168
Pages containing text: 163

--- First page preview ---

CONTENTS
1
8
2
9
3
10
4
11
5
12
6
13
7
14
SERVING THE POOR AND MARGINALISED
NATIONAL SECURITY AND FOREIGN POLICY
ENSURING FARMER WELFARE
ECONOMY AND REFORMS
NARI SHAKTI FOR NEW INDIA
EASE OF DOING BUSINESS
YOUTH-LED DEVELOPMENT
NORTHEAST – A GROWTH ENGINE
MIDDLE CLASS AND EASE OF LIVING
CONSERVING CULTURAL HERITAGE
INFRA AT SPEED AND SCALE
TECH-POWERED INDIA
HEALTHCARE AND PANDEMIC MANAGEMENT
ENVIRONMENT AND SUSTAINABILITY
1
15
27
39
47
59
71
81
93
103
113
123
135
145


In [8]:
def fixed_size_chunks(documents, chunk_size=1000, overlap=200):
    chunks = []

    for doc in documents:
        text = doc["text"]
        page = doc["page"]

        start = 0

        while start < len(text):
            end = start + chunk_size

            chunks.append({
                "text": text[start:end],
                "page": page
            })

            start += chunk_size - overlap

    return chunks


fixed_chunks = fixed_size_chunks(documents)

print("Fixed-size chunks:", len(fixed_chunks))
print("\nExample chunk:\n")
print(fixed_chunks[0]["text"][:1000])
print("\nPage:", fixed_chunks[0]["page"])

Fixed-size chunks: 252

Example chunk:

CONTENTS
1
8
2
9
3
10
4
11
5
12
6
13
7
14
SERVING THE POOR AND MARGINALISED
NATIONAL SECURITY AND FOREIGN POLICY
ENSURING FARMER WELFARE
ECONOMY AND REFORMS
NARI SHAKTI FOR NEW INDIA
EASE OF DOING BUSINESS
YOUTH-LED DEVELOPMENT
NORTHEAST – A GROWTH ENGINE
MIDDLE CLASS AND EASE OF LIVING
CONSERVING CULTURAL HERITAGE
INFRA AT SPEED AND SCALE
TECH-POWERED INDIA
HEALTHCARE AND PANDEMIC MANAGEMENT
ENVIRONMENT AND SUSTAINABILITY
1
15
27
39
47
59
71
81
93
103
113
123
135
145

Page: 5


In [9]:
import re

def sentence_chunks(documents, sentences_per_chunk=5):
    chunks = []

    for doc in documents:
        text = doc["text"]
        page = doc["page"]

        sentences = re.split(r'(?<=[.!?])\s+', text)

        for i in range(0, len(sentences), sentences_per_chunk):
            chunk = " ".join(sentences[i:i + sentences_per_chunk]).strip()

            if chunk:
                chunks.append({
                    "text": chunk,
                    "page": page
                })

    return chunks


sentence_chunks_data = sentence_chunks(documents)

print("Sentence-based chunks:", len(sentence_chunks_data))
print("\nExample chunk:\n")
print(sentence_chunks_data[0]["text"][:1000])
print("\nPage:", sentence_chunks_data[0]["page"])

Sentence-based chunks: 266

Example chunk:

CONTENTS
1
8
2
9
3
10
4
11
5
12
6
13
7
14
SERVING THE POOR AND MARGINALISED
NATIONAL SECURITY AND FOREIGN POLICY
ENSURING FARMER WELFARE
ECONOMY AND REFORMS
NARI SHAKTI FOR NEW INDIA
EASE OF DOING BUSINESS
YOUTH-LED DEVELOPMENT
NORTHEAST – A GROWTH ENGINE
MIDDLE CLASS AND EASE OF LIVING
CONSERVING CULTURAL HERITAGE
INFRA AT SPEED AND SCALE
TECH-POWERED INDIA
HEALTHCARE AND PANDEMIC MANAGEMENT
ENVIRONMENT AND SUSTAINABILITY
1
15
27
39
47
59
71
81
93
103
113
123
135
145

Page: 5


In [10]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [11]:
import numpy as np

fixed_texts = [chunk["text"] for chunk in fixed_chunks]

fixed_embeddings = embedding_model.encode(
    fixed_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Number of embeddings:", len(fixed_embeddings))
print("Embedding dimension:", fixed_embeddings.shape[1])

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Number of embeddings: 252
Embedding dimension: 384


In [12]:
import faiss

dimension = fixed_embeddings.shape[1]

fixed_index = faiss.IndexFlatL2(dimension)
fixed_index.add(fixed_embeddings.astype("float32"))

print("FAISS index created!")
print("Vectors stored:", fixed_index.ntotal)

FAISS index created!
Vectors stored: 252


In [13]:
def retrieve_fixed(question, k=5):
    # Convert the question into an embedding
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    # Search FAISS
    distances, indices = fixed_index.search(question_embedding, k)

    results = []

    for distance, index in zip(distances[0], indices[0]):
        results.append({
            "text": fixed_chunks[index]["text"],
            "page": fixed_chunks[index]["page"],
            "distance": float(distance)
        })

    return results

In [14]:
question = "What is PM-KISAN?"

results = retrieve_fixed(question, k=5)

for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} | Page {result['page']} ---")
    print(result["text"][:800])
    print("Distance:", result["distance"])


--- Result 1 | Page 24 ---
edness. The 
Pradhan Mantri Fasal Bima 
Yojana changed this status quo. 
Launched in 2016, the scheme 
overhauled the agriculture 
insurance system and has 
provided a much-needed safety 
net for farmers. The PM Krishi 
Sinchayee Yojana has expanded 
irrigation coverage and reduced 
farmers’ dependence on rainfall 
for irrigation.
Record production 
and consequent increase 
in procurement by the 
government have led to an 
unprecedented rise in farm 
incomes. Moreover, the gradual 
transition towards payment 
to farmers through DBT has 
protected farmers from unfair 
charges and commissions 
charged by intermediaries. 
In 2019, the government 
launched the PM Kisan 
Samman Nidhi to provide an 
assured cash transfer of Rs. 
6,000 per year to farmers. This 
supplemental income received 
thro
Distance: 1.035199761390686

--- Result 2 | Page 134 ---
ers at PM 
Sangrahalaya.
* * *
Distance: 1.1157569885253906

--- Result 3 | Page 36 ---
overcome 
social barriers 

In [15]:
sentence_texts = [chunk["text"] for chunk in sentence_chunks_data]

sentence_embeddings = embedding_model.encode(
    sentence_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Sentence-based embeddings:", len(sentence_embeddings))
print("Embedding dimension:", sentence_embeddings.shape[1])

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Sentence-based embeddings: 266
Embedding dimension: 384


In [16]:
sentence_index = faiss.IndexFlatL2(sentence_embeddings.shape[1])

sentence_index.add(
    sentence_embeddings.astype("float32")
)

print("Sentence-based FAISS index created!")
print("Vectors stored:", sentence_index.ntotal)

Sentence-based FAISS index created!
Vectors stored: 266


In [17]:
def retrieve_sentence(question, k=5):
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = sentence_index.search(
        question_embedding,
        k
    )

    results = []

    for distance, index in zip(distances[0], indices[0]):
        results.append({
            "text": sentence_chunks_data[index]["text"],
            "page": sentence_chunks_data[index]["page"],
            "distance": float(distance)
        })

    return results

In [18]:
question = "What is PM-KISAN?"

results_sentence = retrieve_sentence(question, k=5)

for i, result in enumerate(results_sentence, 1):
    print(f"\n--- Result {i} | Page {result['page']} ---")
    print(result["text"][:800])
    print("Distance:", result["distance"])


--- Result 1 | Page 36 ---
One Stop Centres 
(OSCs) assist women in 
distress. The Pradhan Mantri 
Matru Vandana Yojana 
(PMMVY) and Pradhan 
Mantri Surakshit Matritva 
Abhiyan (PMSMA) incentivize 
institutional deliveries and 
regular health check-ups for 
pregnant women and lactating 
mothers. The impact of this 
approach has been evident in 
many outcomes, such as an 
improved sex ratio at birth, a 
rise in institutional deliveries, 
declining infant mortality, and 
a lower maternal mortality 
rate. Programmes such as 
MUDRA Yojana have allowed 
crores of women across the 
country to pursue micro-level 
entrepreneurship and become 
financially independent. PM Modi has made 
unshackling women and freeing 
them from the drudgery of 
everyday life a key objective of 
policymaking.
Distance: 1.1367127895355225

--- Result 2 | Page 24 ---
In 2019, the government 
launched the PM Kisan 
Samman Nidhi to provide an 
assured cash transfer of Rs. 6,000 per year to farmers. This 
supplemental

In [19]:
evaluation_questions = [
    {
        "question": "What is PM-KISAN?",
        "answer": "",
        "expected_page": None
    },
    {
        "question": "What are the benefits provided under PM-KISAN?",
        "answer": "",
        "expected_page": None
    },
    {
        "question": "Who is eligible for PM-KISAN?",
        "answer": "",
        "expected_page": None
    },
    {
        "question": "What is Ayushman Bharat?",
        "answer": "",
        "expected_page": None
    },
    {
        "question": "What benefits are provided under Ayushman Bharat?",
        "answer": "",
        "expected_page": None
    }
]

print("Number of test questions:", len(evaluation_questions))

for i, item in enumerate(evaluation_questions, 1):
    print(f"{i}. {item['question']}")

Number of test questions: 5
1. What is PM-KISAN?
2. What are the benefits provided under PM-KISAN?
3. Who is eligible for PM-KISAN?
4. What is Ayushman Bharat?
5. What benefits are provided under Ayushman Bharat?


In [20]:
for i, item in enumerate(evaluation_questions, 1):

    question = item["question"]

    fixed_result = retrieve_fixed(question, k=1)[0]
    sentence_result = retrieve_sentence(question, k=1)[0]

    print(f"\n{'='*70}")
    print(f"Question {i}: {question}")

    print(f"\nFixed-size → Page {fixed_result['page']}")
    print(fixed_result["text"][:400])

    print(f"\nSentence-based → Page {sentence_result['page']}")
    print(sentence_result["text"][:400])


Question 1: What is PM-KISAN?

Fixed-size → Page 24
edness. The 
Pradhan Mantri Fasal Bima 
Yojana changed this status quo. 
Launched in 2016, the scheme 
overhauled the agriculture 
insurance system and has 
provided a much-needed safety 
net for farmers. The PM Krishi 
Sinchayee Yojana has expanded 
irrigation coverage and reduced 
farmers’ dependence on rainfall 
for irrigation.
Record production 
and consequent increase 
in procurement by the 


Sentence-based → Page 36
One Stop Centres 
(OSCs) assist women in 
distress. The Pradhan Mantri 
Matru Vandana Yojana 
(PMMVY) and Pradhan 
Mantri Surakshit Matritva 
Abhiyan (PMSMA) incentivize 
institutional deliveries and 
regular health check-ups for 
pregnant women and lactating 
mothers. The impact of this 
approach has been evident in 
many outcomes, such as an 
improved sex ratio at birth, a 
rise in institutional

Question 2: What are the benefits provided under PM-KISAN?

Fixed-size → Page 24
edness. The 
Pradhan Mantri Fasal Bim

In [21]:
for i, item in enumerate(evaluation_questions, 1):
    question = item["question"]

    fixed = retrieve_fixed(question, k=3)
    sentence = retrieve_sentence(question, k=3)

    print("\n" + "=" * 80)
    print(f"QUESTION {i}: {question}")

    print("\nFIXED-SIZE TOP RESULT")
    print("Page:", fixed[0]["page"])
    print(fixed[0]["text"][:1000])

    print("\nSENTENCE-BASED TOP RESULT")
    print("Page:", sentence[0]["page"])
    print(sentence[0]["text"][:1000])


QUESTION 1: What is PM-KISAN?

FIXED-SIZE TOP RESULT
Page: 24
edness. The 
Pradhan Mantri Fasal Bima 
Yojana changed this status quo. 
Launched in 2016, the scheme 
overhauled the agriculture 
insurance system and has 
provided a much-needed safety 
net for farmers. The PM Krishi 
Sinchayee Yojana has expanded 
irrigation coverage and reduced 
farmers’ dependence on rainfall 
for irrigation.
Record production 
and consequent increase 
in procurement by the 
government have led to an 
unprecedented rise in farm 
incomes. Moreover, the gradual 
transition towards payment 
to farmers through DBT has 
protected farmers from unfair 
charges and commissions 
charged by intermediaries. 
In 2019, the government 
launched the PM Kisan 
Samman Nidhi to provide an 
assured cash transfer of Rs. 
6,000 per year to farmers. This 
supplemental income received 
through regular instalments has 
enabled even small farmers to 
purchase better quality inputs, 
improve their land productivity, 
and increa

In [23]:
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

print("API key loaded:", api_key is not None)

API key loaded: True


In [24]:
!pip install -q google-genai

In [25]:
from google import genai
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

print("Gemini connected successfully!")

Gemini connected successfully!


In [28]:
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Say hello in one sentence."
)

print(response.text)

Hello, and I hope you are having a wonderful day!


In [29]:
import os

pdf_files = [f for f in os.listdir("/content") if f.lower().endswith(".pdf")]

print("PDF files found:")
for f in pdf_files:
    print("-", f)

print("\nTotal PDFs:", len(pdf_files))

PDF files found:
- English_Booklet_2.pdf

Total PDFs: 1


In [31]:
!pip install -q pypdf

from pypdf import PdfReader

pdf_path = "/content/English_Booklet_2.pdf"

reader = PdfReader(pdf_path)

print("Number of pages:", len(reader.pages))

# Extract text from all pages
pages = []

for i, page in enumerate(reader.pages):
    text = page.extract_text() or ""
    pages.append({
        "page": i + 1,
        "text": text
    })

print("Pages extracted:", len(pages))
print("\nFirst page preview:")
print(pages[0]["text"][:1000])

Number of pages: 168
Pages extracted: 168

First page preview:



In [32]:
# Create chunks from the extracted pages

chunks = []

chunk_size = 1000
overlap = 200

for page_data in pages:
    text = page_data["text"].strip()
    page_number = page_data["page"]

    if not text:
        continue

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "page": page_number,
                "text": chunk_text
            })

        start += chunk_size - overlap

print("Total chunks created:", len(chunks))

print("\nExample chunk:")
print(chunks[0]["text"][:1000])
print("\nSource page:", chunks[0]["page"])

Total chunks created: 252

Example chunk:
CONTENTS
1
8
2
9
3
10
4
11
5
12
6
13
7
14
SERVING THE POOR AND MARGINALISED
NATIONAL SECURITY AND FOREIGN POLICY
ENSURING FARMER WELFARE
ECONOMY AND REFORMS
NARI SHAKTI FOR NEW INDIA
EASE OF DOING BUSINESS
YOUTH-LED DEVELOPMENT
NORTHEAST – A GROWTH ENGINE
MIDDLE CLASS AND EASE OF LIVING
CONSERVING CULTURAL HERITAGE
INFRA AT SPEED AND SCALE
TECH-POWERED INDIA
HEALTHCARE AND PANDEMIC MANAGEMENT
ENVIRONMENT AND SUSTAINABILITY
1
15
27
39
47
59
71
81
93
103
113
123
135
145

Source page: 5
